# WDLadder: scaled weight decay on an MAI-style MoE scaling ladder (orchestrator)
Run this notebook on the **compute instance `cpurlvr`** (kernel *Python 3.10 - SDK v2*), one cell at a time.
Every step ends with **CHECK BEFORE CONTINUING**. If a check fails, stop and fix it, because later steps cost money.

| Step | Where | Cost |
|---|---|---|
| 0-4 setup, unit + local smoke tests | instance | ~$1 |
| 5-6 full data pool + upload | instance | ~$3 |
| 7 cluster smoke (pre-emption/resume on CPU + GPU) | clusters | ~$0.3 |
| 8 benchmarks -> real budget table | clusters | ~$0.5 |
| 9-10 stage 1 -> **GATE 1** | CPU cluster | ~$4 |
| 11-14 stages 2-6 (each gated) | CPU + T4 | ~$40-135 |

**Stop the compute instance whenever you are not using it** ($1.31/h). Cluster jobs keep running without it.
Use the check-in cell (9b) once or twice a day to resubmit pre-empted runs; each check-in takes about 1 minute.

In [ ]:
# ===== H (v2) =====
import os, sys, json, glob, shutil, importlib, importlib.util, nbformat
from IPython import get_ipython
W = "/mnt/batch/tasks/shared/LS_root/mounts/clusters/cpurlvr/code/Users/estherxin0011/wdladder"
ALT = "/home/azureuser/cloudfiles/code/Users/estherxin0011/wdladder"
os.chdir(W); sys.path.insert(0, W); importlib.invalidate_caches()
if not os.path.exists(W + "/chain.py"):
    src = next((p for d in (ALT, W, os.path.dirname(ALT)) for p in sorted(glob.glob(d + "/*chain*.py"))), None)
    assert src, "chain.py not found in " + ALT + " or " + W
    shutil.copy(src, W + "/chain.py"); print("copied", src, "->", W + "/chain.py")
import config as C, azure_jobs as AJ
spec = importlib.util.spec_from_file_location("chain", W + "/chain.py")
chain = importlib.util.module_from_spec(spec); sys.modules["chain"] = chain; spec.loader.exec_module(chain)
print("chain loaded:", chain.__file__, os.path.getsize(chain.__file__), "bytes")
PY = sys.executable
def NB(key, i=None):
    nb = nbformat.read("orchestrate.ipynb", 4); md = ""; hits = []
    for c in nb.cells:
        if c.cell_type == "markdown": md = c.source; continue
        if key.lower() in (md + "\n" + c.source).lower(): hits.append(c.source)
    if not hits: return print("no cell matches", key)
    if i is None and len(hits) > 1:
        for k, s in enumerate(hits): print(f"--- [{k}]\n{s[:300]}\n")
        return print(f"{len(hits)} matches -> NB({key!r}, i=K)")
    get_ipython().run_cell(hits[i or 0])
def DONE(stages, l6_tpp=120):
    st = {r["name"]: chain._remote_state(r["name"]) for r in AJ.stage_runs(stages, l6_tpp)}
    print({s: list(st.values()).count(s) for s in set(st.values())}); return st



In [ ]:
import glob, json, re, shutil, numpy as np
R = "/home/azureuser/wdladder_data/runs"; X = "/home/azureuser/wdladder_data/runs_excluded"
AJ.sync(); os.makedirs(X, exist_ok=True)
for d in glob.glob(R + "/*smoke*") + glob.glob(R + "/bench_*"):
    shutil.rmtree(os.path.join(X, os.path.basename(d)), ignore_errors=True); shutil.move(d, X)
pat = re.compile(r"^L(\d+)_t120_cos_(const|scaled)_lr([\d.]+)_p([\d.]+)_s(\d+)$")
D = {}
for d in glob.glob(R + "/L*_t120_cos_*"):
    m = pat.match(os.path.basename(d))
    if not m or float(m[3]) != 1.0: continue
    try: s = json.load(open(d + "/summary.json"))
    except Exception: continue
    if s.get("status") == "done":
        D.setdefault((int(m[1]), m[2]), {}).setdefault(float(m[4]), []).append(s["final"]["bpb"])

def fit(pts):
    ps = sorted(pts); x = np.log2(ps); y = np.array([np.mean(pts[p]) for p in ps])
    if len(ps) >= 3:
        k = np.polyfit(x, y, 2); a, b = k[0], k[1]
        xm = float(np.clip(-b / (2 * a), x.min() - 1, x.max() + 1)) if a > 0 else float(x[y.argmin()])
        ymin = min(float(np.polyval(k, xm)), float(y.min()))
        y1 = float(np.mean(pts[1.0])) if 1.0 in pts else float(np.polyval(k, 0.0))
    else:
        xm, ymin = float(x[y.argmin()]), float(y.min())
        y1 = float(np.mean(pts[1.0])) if 1.0 in pts else float(np.interp(0.0, x, y))
    edge = not (x.min() <= xm <= x.max())
    return dict(n=len(ps), p_opt=round(2 ** xm, 3), bpb_opt=ymin, bpb_p1=y1, edge=edge)

print(f"{'L':>3} {'untuned gap':>12} {'tuned gap':>10} {'survival':>9} {'p_opt c/s':>16} {'WD ratio':>9}  edge(c,s)  npts(c,s)")
for L in sorted({k[0] for k in D}):
    if (L, "const") not in D or (L, "scaled") not in D: continue
    c, s = fit(D[(L, "const")]), fit(D[(L, "scaled")])
    ug, tg = c["bpb_p1"] - s["bpb_p1"], c["bpb_opt"] - s["bpb_opt"]
    print(f"{L:>3} {ug:12.5f} {tg:10.5f} {tg/ug if ug > 0 else float('nan'):9.3f} "
          f"{c['p_opt']:>7}/{s['p_opt']:<7} {s['p_opt']/c['p_opt']:9.2f}  {c['edge']!s:5},{s['edge']!s:5}  {c['n']},{s['n']}")
for k in sorted(D): print(k, {p: round(float(np.mean(v)), 5) for p, v in sorted(D[k].items())})

In [ ]:
!{PY} analysis.py --stage 5
!{PY} llm_review.py review --stage 5
chain.stop("gpu"); chain.status("gpu", runs=False)

In [ ]:
import glob, json, re, numpy as np
R = "/home/azureuser/wdladder_data/runs"
pat = re.compile(r"^L(\d+)_t120_cos_(const|scaled)_lr([\d.]+)_p([\d.]+)_s(\d+)$")
D = {}
for d in glob.glob(R + "/L*_t120_cos_*"):
    m = pat.match(os.path.basename(d))
    if not m or float(m[3]) != 1.0: continue
    try: s = json.load(open(d + "/summary.json"))
    except Exception: continue
    if s.get("status") == "done":
        D.setdefault((int(m[1]), m[2]), {}).setdefault(float(m[4]), []).append(s["final"]["bpb"])
def fit(pts):
    ps = sorted(pts); x = np.log2(ps); y = np.array([np.mean(pts[p]) for p in ps])
    if len(ps) >= 3:
        k = np.polyfit(x, y, 2); a, b = k[0], k[1]
        xm = float(np.clip(-b/(2*a), x.min()-1, x.max()+1)) if a > 0 else float(x[y.argmin()])
        ymin = min(float(np.polyval(k, xm)), float(y.min()))
        y1 = float(np.mean(pts[1.0])) if 1.0 in pts else float(np.polyval(k, 0.0))
    else:
        xm, ymin = float(x[y.argmin()]), float(y.min())
        y1 = float(np.mean(pts[1.0])) if 1.0 in pts else float(np.interp(0.0, x, y))
    return dict(n=len(ps), p_opt=round(2**xm, 3), bpb_opt=ymin, bpb_p1=y1)
print(f"{'L':>3} {'untuned gap':>12} {'tuned gap':>10} {'survival':>9} {'p_opt c/s':>16} {'WD ratio':>9}")
for L in sorted({k[0] for k in D}):
    if (L, "const") not in D or (L, "scaled") not in D: continue
    c, s = fit(D[(L, "const")]), fit(D[(L, "scaled")])
    ug, tg = c["bpb_p1"] - s["bpb_p1"], c["bpb_opt"] - s["bpb_opt"]
    print(f"{L:>3} {ug:12.5f} {tg:10.5f} {tg/ug if ug > 0 else float('nan'):9.3f} {c['p_opt']:>7}/{s['p_opt']:<7} {s['p_opt']/c['p_opt']:9.2f}")
for k in sorted(D): print(k, {p: round(float(np.mean(v)), 5) for p, v in sorted(D[k].items())})

In [ ]:
import glob, json
R = "/home/azureuser/wdladder_data/runs"
AJ.sync()
L6 = {}
for d in sorted(glob.glob(R + "/L6_*")):
    n = d.split("/")[-1]
    try: s = json.load(open(d + "/summary.json")); L6[n] = s["final"]["bpb"] if s.get("status") == "done" else None
    except Exception: L6[n] = None
    p = chain._remote_state(n); print(f"{n:45s} {p:15s} bpb={L6[n]}")
def get(meth, untuned):
    return next((v for k, v in L6.items() if f"_{meth}_" in k and (("_lr1_p1_" in k) == untuned)), None)
tc, ts, uc, us = get("const", False), get("scaled", False), get("const", True), get("scaled", True)
if tc and ts: print(f"L6 tuned gap   = {tc-ts:.5f}   (L3 tuned gap 0.00261)")
if uc and us: print(f"L6 untuned gap = {uc-us:.5f}   (L3 untuned gap 0.03499)")
if tc and ts and uc and us: print(f"L6 survival    = {(tc-ts)/(uc-us):.3f}   (L3 0.074)")
if not ((tc and ts) or (uc and us)): print("no complete pair yet -> wait")

In [ ]:
import shutil, glob
def CLEAN():
    AJ.sync()
    R = "/home/azureuser/wdladder_data/runs"; X = "/home/azureuser/wdladder_data/runs_excluded"
    os.makedirs(X, exist_ok=True)
    for d in glob.glob(R + "/*smoke*") + glob.glob(R + "/bench_*"):
        shutil.rmtree(os.path.join(X, os.path.basename(d)), ignore_errors=True); shutil.move(d, X)
    print(len([d for d in os.listdir(R) if not d.startswith("_")]), "runs")
CLEAN()
!{PY} analysis.py --stage 3

In [ ]:
import glob, json, statistics as S
R = "/home/azureuser/wdladder_data/runs"
s0 = json.load(open(sorted(glob.glob(R + "/L3_t120_cos_*_s1/summary.json"))[0]))
print("summary keys:", list(s0))
def num(v):
    return v if isinstance(v, (int, float)) else (v.get("all", v.get("total", v.get("mean"))) if isinstance(v, dict) else None)
KEY = next((k for k in ("bpb", "final_bpb", "eval_bpb", "bpb_all", "final_eval_bpb", "val_bpb") if num(s0.get(k)) is not None),
           next((k for k, v in s0.items() if "bpb" in k.lower() and num(v) is not None), None))
print("using key:", KEY)
def bpb(n):
    try: return num(json.load(open(f"{R}/{n}/summary.json")).get(KEY))
    except Exception: return None
T = {}
for d in sorted(glob.glob(R + "/L3_t120_cos_*_s[1-4]")):
    n = d.split("/")[-1]; T.setdefault(n.rsplit("_s", 1)[0], {})[int(n.rsplit("_s", 1)[1])] = bpb(n)
T = {k: v for k, v in T.items() if len(v) >= 3}
for k, v in T.items(): print(k, v)
c = next((v for k, v in T.items() if "_const_" in k), {}); s = next((v for k, v in T.items() if "_scaled_" in k), {})
d = [c[i] - s[i] for i in sorted(set(c) & set(s)) if c[i] is not None and s[i] is not None]
TC = {2: 12.706, 3: 4.303, 4: 3.182, 5: 2.776, 6: 2.571}
if len(d) >= 2:
    m = S.mean(d); se = S.stdev(d) / len(d) ** 0.5; t = TC.get(len(d), 2.0)
    print(f"paired gap {m:.5f}  95% CI [{m-t*se:.5f}, {m+t*se:.5f}]  n={len(d)}  per-seed {[round(x,5) for x in d]}")
    print("survival vs untuned 0.035:", round(m / 0.03499458, 3), "| CI above 0.002 null:", m - t*se > 0.002)
else:
    print("not enough paired values:", d)
!{PY} analysis.py --stage 4

In [ ]:
import glob, json, statistics as S
R = "/home/azureuser/wdladder_data/runs"
s0 = json.load(open(sorted(glob.glob(R + "/L3_t120_cos_*_s1/summary.json"))[0]))
print("final =", json.dumps(s0["final"])[:600])
def flat(v, p=""):
    if isinstance(v, dict):
        for k, x in v.items(): yield from flat(x, f"{p}.{k}" if p else k)
    elif isinstance(v, (int, float)): yield p, v
F = dict(flat(s0["final"]))
KEY = next((k for k in ("bpb", "all", "bpb.all", "total", "bpb.total", "mean", "bpb.mean") if k in F),
           next((k for k in F if "bpb" in k.lower()), None))
print("using final key:", KEY)
def bpb(n):
    try: return dict(flat(json.load(open(f"{R}/{n}/summary.json"))["final"])).get(KEY)
    except Exception: return None
T = {}
for d in sorted(glob.glob(R + "/L3_t120_cos_*_s[1-4]")):
    n = d.split("/")[-1]; T.setdefault(n.rsplit("_s", 1)[0], {})[int(n.rsplit("_s", 1)[1])] = bpb(n)
T = {k: v for k, v in T.items() if len(v) >= 3}
for k, v in T.items(): print(k, v)
c = next((v for k, v in T.items() if "_const_" in k), {}); s = next((v for k, v in T.items() if "_scaled_" in k), {})
d = [c[i] - s[i] for i in sorted(set(c) & set(s)) if c[i] is not None and s[i] is not None]
TC = {2: 12.706, 3: 4.303, 4: 3.182, 5: 2.776}
if len(d) >= 2:
    m = S.mean(d); se = S.stdev(d) / len(d) ** 0.5; t = TC.get(len(d), 2.0)
    print(f"paired gap {m:.5f}  95% CI [{m-t*se:.5f}, {m+t*se:.5f}]  n={len(d)}  per-seed {[round(x,5) for x in d]}")
    print("survival vs untuned 0.035:", round(m / 0.03499458, 3), "| CI lower bound > 0.002 null:", m - t*se > 0.002)
else:
    print("not enough paired values:", d)

In [ ]:
import shutil, glob
def CLEAN():
    AJ.sync()
    R = "/home/azureuser/wdladder_data/runs"; X = "/home/azureuser/wdladder_data/runs_excluded"
    os.makedirs(X, exist_ok=True)
    for d in glob.glob(R + "/*smoke*") + glob.glob(R + "/bench_*"):
        shutil.rmtree(os.path.join(X, os.path.basename(d)), ignore_errors=True); shutil.move(d, X)
    print(len([d for d in os.listdir(R) if not d.startswith("_")]), "runs")
CLEAN()
!{PY} analysis.py --stage 3
!{PY} llm_review.py review --stage 3

In [ ]:
import plan, json
def GO(stages, l6=120):
    chain.add([dict(r, device="gpu") for r in AJ.stage_runs(stages, l6)], cid="gpu")
    return DONE(stages, l6)
T = [r for r in AJ.stage_runs([3], 120) if r["sched"] == "cos" and r["seed"] in (1, 2)]
EXTRA = [dict(plan.spec(3, r["L"], r["tpp"], "cos", r["method"], r["lr_mult"], r["prod_mult"], s), device="gpu")
         for r in T if r["seed"] == 1 for s in (3, 4)]
R5 = AJ.stage_runs([5], 120); s0 = min(r["seed"] for r in R5)
K5 = [dict(r, device="gpu") for r in R5 if r["seed"] == s0]
print("extra seeds:", [r["name"] for r in EXTRA])
print("L6:", len(R5), "->", len(K5), [r["name"] for r in K5])
chain.add(EXTRA + K5, cid="gpu")
R = "/home/azureuser/wdladder_data/runs"
for r in sorted([r for r in AJ.stage_runs([3], 120) if r["sched"] == "wsd"], key=lambda r: (r["method"], r["prod_mult"])):
    s = json.load(open(f"{R}/{r['name']}/summary.json"))
    print(f"{r['method']:7s} prod={r['prod_mult']:<6} bpb={s.get('bpb', s.get('final_bpb'))}")

In [ ]:
import shutil, glob
def CLEAN():
    AJ.sync()
    R = "/home/azureuser/wdladder_data/runs"; X = "/home/azureuser/wdladder_data/runs_excluded"
    os.makedirs(X, exist_ok=True)
    for d in glob.glob(R + "/*smoke*") + glob.glob(R + "/bench_*"):
        shutil.rmtree(os.path.join(X, os.path.basename(d)), ignore_errors=True); shutil.move(d, X)
    print(len([d for d in os.listdir(R) if not d.startswith("_")]), "runs")
CLEAN()
!{PY} analysis.py --stage 2
!{PY} llm_review.py review --stage 2

In [ ]:
def GO(stages, l6=120):
    chain.add([dict(r, device="gpu") for r in AJ.stage_runs(stages, l6)], cid="gpu")
    return DONE(stages, l6)
print({s: len(AJ.stage_runs([s], 120)) for s in (3, 4)})
GO([3, 4])

In [ ]:
import shutil, glob
def CLEAN():
    AJ.sync()
    R = "/home/azureuser/wdladder_data/runs"; X = "/home/azureuser/wdladder_data/runs_excluded"
    os.makedirs(X, exist_ok=True)
    for d in glob.glob(R + "/*smoke*") + glob.glob(R + "/bench_*"):
        shutil.rmtree(os.path.join(X, os.path.basename(d)), ignore_errors=True); shutil.move(d, X)
    print(len([d for d in os.listdir(R) if not d.startswith("_")]), "runs")
CLEAN()
!{PY} analysis.py --stage 2
!{PY} llm_review.py review --stage 2

In [ ]:
def GO(stages, l6=120):
    chain.add([dict(r, device="gpu") for r in AJ.stage_runs(stages, l6)], cid="gpu")
    return DONE(stages, l6)
print(len(AJ.stage_runs([2], 120)), "stage-2 runs")
GO([2])

In [ ]:
chain.status("gpu", runs=False); DONE([2])

In [ ]:
import shutil, glob
def CLEAN():
    AJ.sync()
    R = "/home/azureuser/wdladder_data/runs"; X = "/home/azureuser/wdladder_data/runs_excluded"
    os.makedirs(X, exist_ok=True)
    for d in glob.glob(R + "/*smoke*") + glob.glob(R + "/bench_*"):
        shutil.rmtree(os.path.join(X, os.path.basename(d)), ignore_errors=True); shutil.move(d, X)
    print(len([d for d in os.listdir(R) if not d.startswith("_")]), "runs")
CLEAN()
!{PY} analysis.py --stage 2
!{PY} llm_review.py review --stage 2

In [ ]:
def GO(stages, l6=120):
    chain.add([dict(r, device="gpu") for r in AJ.stage_runs(stages, l6)], cid="gpu")
    return DONE(stages, l6)
chain.status("gpu"); DONE([1])

In [ ]:
import shutil, glob
AJ.sync()
R = "/home/azureuser/wdladder_data/runs"; X = "/home/azureuser/wdladder_data/runs_excluded"
os.makedirs(X, exist_ok=True)
for d in glob.glob(R + "/*smoke*") + glob.glob(R + "/bench_*"):
    shutil.rmtree(os.path.join(X, os.path.basename(d)), ignore_errors=True); shutil.move(d, X)
print(sorted(os.listdir(R)))
!{PY} analysis.py --stage 1
!{PY} llm_review.py review --stage 1

In [ ]:
def GO(stages, l6=120):
    chain.add([dict(r, device="gpu") for r in AJ.stage_runs(stages, l6)], cid="gpu")
    return DONE(stages, l6)
GO([2])

In [ ]:
AJ.sync()
!{PY} analysis.py --stage 1
!{PY} llm_review.py review --stage 1

In [ ]:
import time
ml = AJ.client(); J = ["silly_oregano_xy0bmjk3rw", "epic_bucket_cffmcd7sn1", "crimson_jelly_4gdpdmmrbq"]
for j in J: ml.jobs.begin_cancel(j)
time.sleep(90); print({j: ml.jobs.get(j).status for j in J})
cont = AJ.container()
def CLEAR(runs):
    for r in runs:
        bl = [b.name for b in cont.list_blobs(name_starts_with=f"{C.AZURE['remote_prefix']}/runs/{r['name']}/")]
        for b in bl: cont.delete_blob(b)
        print(r["name"], "cleared", len(bl))
R1 = AJ.stage_runs([1], 120); CLEAR(R1)
chain.add([dict(r, device="gpu") for r in R1], cid="gpu")

In [ ]:
chain.status("bench", runs=False)
for n in [f"bench_cpu_L{L}" for L in (3, 4)] + [f"bench_gpu_L{L}" for L in (3, 4, 6, 9)]:
    print(n, AJ.read_remote_json(n, "bench.json"))

In [ ]:
AJ.sync()
!{PY} plan.py --budget

In [ ]:
chain.checkin([1], l6_tpp=120)

In [ ]:
import plan
mk = lambda dev, L: dict(plan.spec(0, L, 120, "cos", "const", 1, 1, 0), name=f"bench_{dev}_L{L}", device=dev)
print([(r["name"], r["L"]) for r in [mk("cpu", 3), mk("gpu", 9)]])
print(AJ.submit([mk("cpu", L) for L in (3, 4)], pool_asset="azureml:wdladder-pool:1",
                extra_args="--benchmark 20", usd_override=0.08, tag_stage="bench"))
chain.add([mk("gpu", L) for L in (3, 4, 6, 9)], cid="bench", extra="--benchmark 20")

In [ ]:
import os, sys, glob, shutil, importlib, importlib.util
W = "/mnt/batch/tasks/shared/LS_root/mounts/clusters/cpurlvr/code/Users/estherxin0011/wdladder"
ALT = "/home/azureuser/cloudfiles/code/Users/estherxin0011/wdladder"
os.chdir(W); sys.path.insert(0, W); importlib.invalidate_caches()
src = next((p for d in (W, ALT) for p in sorted(glob.glob(d + "/chain_v2*.py"))), None)
assert src, "upload chain_v2.py into wdladder first"
shutil.copy(src, W + "/chain.py"); os.remove(src)
import config as C, azure_jobs as AJ
spec = importlib.util.spec_from_file_location("chain", W + "/chain.py")
chain = importlib.util.module_from_spec(spec); sys.modules["chain"] = chain; spec.loader.exec_module(chain)
assert hasattr(chain, "reset"); print("chain loaded", chain.__file__, "v2 OK")
chain.reset("smoke")
base = dict(AJ.stage_runs([1], 120)[0])
chain.add([dict(base, name="chain_smoke_gpu"), dict(base, name="chain_smoke_gpu2")], cid="smoke",
          pool_asset="azureml:wdladder-pool:smoke", extra="--smoke --steps 300 --stop-at-step 150", test_evict=True)
chain.status("bench", runs=False)

## 0. Setup: code location, packages, secrets
Upload all `.py` files plus `requirements.txt`, `.amlignore` and `tests/smoke_test.py` into
`/home/azureuser/cloudfiles/code/Users/estherxin0011/wdladder/`, keeping the `tests/` sub-folder.
The cell asks for your keys once and stores them in `~/.wdladder_secrets` (chmod 600, never in code).
**Rotate the Foundry key and HF token that were pasted in chat before you start.**

**CHECK BEFORE CONTINUING:** `torch 2.4.1+cpu` is printed and `secrets file OK`.

In [ ]:
!pip install --upgrade pip

In [ ]:
import os, sys, getpass, importlib
ROOT = "/home/azureuser/cloudfiles/code/Users/estherxin0011"
PROJECT = f"{ROOT}/wdladder"
os.environ["WDL_PROJECT"] = PROJECT
os.environ["WDL_DATA"] = "/home/azureuser/wdladder_data"
os.chdir(PROJECT); sys.path.insert(0, PROJECT)
PY = sys.executable                      # the kernel's own python
print("kernel python:", PY, sys.version.split()[0])
!{PY} -m pip install -q -r requirements.txt
!{PY} -m pip install -q "torch>=2.4" --index-url https://download.pytorch.org/whl/cpu
importlib.invalidate_caches()
import torch; print("torch", torch.__version__)

SEC = os.path.expanduser("~/.wdladder_secrets")
have = open(SEC).read() if os.path.exists(SEC) else ""
lines = [l for l in have.splitlines() if l.strip()]
for k in ("HF_TOKEN", "AZURE_OPENAI_KEY"):
    if f"{k}=" not in have:
        lines.append(f"{k}=" + getpass.getpass(f"{k}: ").strip())
open(SEC, "w").write("\n".join(lines) + "\n"); os.chmod(SEC, 0o600)
import config as C
print("secrets file OK" if C.secret("HF_TOKEN", False) and C.secret("AZURE_OPENAI_KEY", False) else "MISSING SECRET")

## 1. Unit smoke tests (CPU, ~2 min, $0)
These check MAI law values, the Newton-Schulz orthogonality, dropless MoE routing, decay semantics, data-stream
determinism and the analysis pipeline on a synthetic ladder.

**CHECK BEFORE CONTINUING:**
- `ALL SMOKE TESTS PASSED` is printed.
- The analysis selftest recovers a WD ratio close to the true synthetic value (about 1.2).

In [ ]:
!python config.py
!python tests/smoke_test.py
!python datastream.py --selftest
!python analysis.py --selftest
!python plan.py --budget

## 2. Azure resources: CPU cluster, GPU cluster check, job environment (~10 min first time, $0)
This creates `cpu-rlvr` (F16s_v2, **low priority**, 0 to 6 nodes, scale-down after 10 min idle) and checks `rlvr2`
(T4). It also builds the environment `wdladder-env` (torch 2.4.1 + CUDA 11.8).

**CHECK BEFORE CONTINUING:**
- The GPU cluster shows `min 0`. If min is above 0, the T4 bills 24/7: set it to 0 in the portal.
- Note the GPU **tier**. If it is `low_priority`, set `PRICES["gpu_node"]` to about 0.11 in `config.py`; dedicated is 0.526.
- If you have no low-priority CPU quota, set `AZURE["cpu_low_priority"]=False`. CPU cost rises about 5x, so run more L3 work on the instance instead.
- The environment build succeeds (Azure ML studio > Environments > wdladder-env shows *Succeeded* after the first job).

In [ ]:
!{PY} patch_step2.py
import importlib, json, config as C, azure_jobs as AJ, plan, analysis
importlib.reload(C); importlib.reload(AJ); importlib.reload(plan)
AJ.ensure_cpu_cluster()          # read-only now
AJ.check_gpu_cluster()
env = AJ.ensure_env(); print(env.name, env.version)

## 3. Data smoke + local train/resume test (~10 min, $0.2)
This builds a tiny pool, then trains 40 steps straight and separately trains 20 steps, simulates pre-emption and resumes.

**CHECK BEFORE CONTINUING:**
- `6 resume OK: straight X == resumed X` is printed, meaning the resumed run is bit-for-bit consistent.
- `finished run skipped` is printed, so re-running a finished run never costs compute again.

In [ ]:
!python data_prep.py --smoke all
!python tests/smoke_test.py --resume

## 4. Foundry (GPT-6.1) connectivity ($0.01)
GPT-6.1 (`gpt-6.1-sol`) is used **only** for three things: anomaly review after each gate, log triage, and
drafting the Results section with automatic number verification. It never takes a gate decision; all gates are
pre-registered in `config.PREREG`. If you deploy a GPT-5-series model for cheap triage, put its name in
`FOUNDRY["deployment_fast"]`.

**CHECK BEFORE CONTINUING:** the reply contains `OK`.

In [ ]:
import llm_review
print(llm_review.chat("Reply with the single word OK.", max_tokens=20))

## 5. Full data pool (~1-2.5 h on 20 vCPU, about $3 of instance time)
This builds about 1.27B unique tokens from FineWeb-Edu, The Stack (Python) and OpenWebMath. The pool size
comes from `plan.pool_requirements()` so that **source scaling** (MAI) holds: every rung repeats each source
the same number of times. Run it in a terminal so the browser can close:
```
cd ~/cloudfiles/code/Users/estherxin0011/wdladder
export WDL_DATA=/home/azureuser/wdladder_data
nohup python data_prep.py all > ~/data_prep.log 2>&1 &
tail -f ~/data_prep.log
```
It is resumable: if the instance stops, run the same command again.

**CHECK BEFORE CONTINUING:**
- Every source shows `OK` and the run ends with `[verify] PASS`.
- If any source is `LOW`, run `python data_prep.py fetch --scale 1.2`, delete `pool/`, then run `python data_prep.py tokenize`.
- Bytes per token are about 3.5-4.5 for web and code.

In [ ]:
import subprocess, os
DATA = "/home/azureuser/wdladder_data"
LOG, PIDF = os.path.expanduser("~/data_prep.log"), os.path.expanduser("~/data_prep.pid")

def alive():
    try:
        pid = int(open(PIDF).read()); os.kill(pid, 0); return pid
    except Exception:
        return None

if alive():
    print("already running, pid", alive())
else:
    env = dict(os.environ, WDL_DATA=DATA, WDL_PROJECT=PROJECT, PYTHONUNBUFFERED="1")
    p = subprocess.Popen([PY, "data_prep.py", "all"], cwd=PROJECT, env=env,
                         stdout=open(LOG, "a"), stderr=subprocess.STDOUT, start_new_session=True)
    open(PIDF, "w").write(str(p.pid))
    print("started pid", p.pid, "log:", LOG)

In [ ]:
pid = alive()
print("RUNNING, pid" if pid else "NOT RUNNING (finished or stopped)", pid or "")
!tail -n 15 ~/data_prep.log
!du -sh /home/azureuser/wdladder_data/* 2>/dev/null; df -h /home/azureuser | tail -1

In [ ]:
print("still running" if alive() else "finished - OK to continue")

In [ ]:
!{PY} data_prep.py verify

In [ ]:
import importlib, os, data_prep as D
os.environ["WDL_DATA"] = "/home/azureuser/wdladder_data"
importlib.reload(D)
_orig = D.requirements
BOOST = {"code": 1.6, "math": 1.4}            # 1.43x / 1.25x needed + margin
def _req(smoke, scale):
    r = dict(_orig(smoke, scale))
    for k, f in BOOST.items():
        if isinstance(r.get(k), (int, float)):
            r[k] = r[k] * f
    return r
print("old targets:", _orig(False, 1.0)); D.requirements = _req; print("new targets:", _req(False, 1.0))
D.fetch(False, 1.0)                           # web: target already met -> skipped
D.requirements = _orig

In [ ]:
import glob, json, hashlib
for src in ("code", "math"):
    seen, dup, n = set(), 0, 0
    for f in sorted(glob.glob(f"/home/azureuser/wdladder_data/raw/{src}/shard_*.jsonl")):
        for line in open(f):
            d = json.loads(line); t = d.get("text") or d.get("content") or line
            h = hashlib.md5(t.encode()).digest(); n += 1
            if h in seen: dup += 1
            seen.add(h)
    print(f"{src}: {n} docs, {dup} duplicates ({100*dup/max(n,1):.2f}%)")

In [ ]:
!rm -rf /home/azureuser/wdladder_data/pool
!WDL_DATA=/home/azureuser/wdladder_data {PY} data_prep.py tokenize
!WDL_DATA=/home/azureuser/wdladder_data {PY} data_prep.py verify
!df -h /home/azureuser | tail -1

In [ ]:
import os, shutil, subprocess, importlib, data_prep as D, config as C, azure_jobs as AJ
DATA = "/home/azureuser/wdladder_data"; os.environ["WDL_DATA"] = DATA
importlib.reload(C); importlib.reload(D); importlib.reload(AJ)

_orig = D.requirements
def _req(smoke, scale):
    r = dict(_orig(smoke, scale)); r["code"] *= 2.1; r["math"] *= 1.4; return r
D.requirements = _req
print("targets:", _req(False, 1.0))
D.fetch(False, 1.0)
D.requirements = _orig

shutil.rmtree(f"{DATA}/pool", ignore_errors=True)
env = dict(os.environ, WDL_DATA=DATA, PYTHONUNBUFFERED="1")
subprocess.run([PY, "data_prep.py", "tokenize"], env=env, check=True)
ok = subprocess.run([PY, "data_prep.py", "verify"], env=env).returncode == 0
!df -h /home/azureuser | tail -1
assert ok, "verify still FAIL - send me the output"

print(AJ.upload_pool(C.data_paths(C.SMOKE_ROOT)["pool"]))
print(AJ.upload_pool(C.P["pool"]))
ml = AJ.client()
for v in ("smoke", "1"):
    d = ml.data.get("wdladder-pool", version=v); print("asset OK:", d.name, d.version)

In [ ]:
!python data_prep.py verify

## 6. Upload pools as data assets (~10 min, $0)
Jobs **download** the pool to node-local disk, so training never reads from network storage.

**CHECK BEFORE CONTINUING:** both assets are listed: `wdladder-pool:smoke` and `wdladder-pool:1`.

In [ ]:
AJ.upload_pool(C.data_paths(C.SMOKE_ROOT)["pool"])
AJ.upload_pool(C.P["pool"])

## 7. Cluster smoke: pre-emption and resume on the real CPU and GPU clusters (~30 min, ~$0.3)
Segment 1 stops at step 150 (exit 75, so Azure marks the job *Failed*; **this is expected**). Segment 2 is a
resubmission that must resume from the checkpoint on the datastore and finish.

**CHECK BEFORE CONTINUING:**
- Both runs print `status done, segments 2`.
- The GPU log shows `device cuda` and a finite bpb.
- Job snapshot size is under 1 MB. If it is larger, `.amlignore` was not uploaded.

In [ ]:
import time, json, importlib, config as C, azure_jobs as AJ, plan
importlib.reload(C); importlib.reload(AJ); importlib.reload(plan)
ml = AJ.client()

def wait(jobs, poll=60):
    while True:
        st = {j: ml.jobs.get(j).status for j in jobs}
        print(time.strftime("%H:%M"), st, flush=True)
        if all(s in ("Completed", "Failed", "Canceled") for s in st.values()):
            return st
        time.sleep(poll)

def adhoc(name, device, L=3):
    return dict(name=name, stage=0, L=L, tpp=120, sched="cos", method="scaled",
                lr_mult=1.0, prod_mult=1.0, seed=0, device=device)

# ---- STEP 7: cluster smoke (segment 1 stops at step 150 -> job shows Failed = expected; segment 2 resumes)
SMOKE = [adhoc("smoke_cpu", "cpu"), adhoc("smoke_gpu", "gpu")]
P = "azureml:wdladder-pool:smoke"
wait(AJ.submit(SMOKE, pool_asset=P, extra_args="--smoke --steps 300 --stop-at-step 150", usd_override=0.05, tag_stage="smoke"))
wait(AJ.submit(SMOKE, pool_asset=P, extra_args="--smoke --steps 300", usd_override=0.05, tag_stage="smoke"))
ok = True
for r in SMOKE:
    s = AJ.read_remote_json(r["name"], "summary.json") or {}
    print(r["name"], "| status", s.get("status"), "| segments", s.get("segments"), "| device", s.get("device"),
          "| bpb", (s.get("final") or {}).get("bpb"))
    ok &= s.get("status") == "done" and s.get("segments") == 2
assert ok, "STEP 7 FAILED - open the job in studio > Jobs > Outputs+logs > user_logs/std_log.txt and send it to me"

# ---- STEP 8: throughput benchmarks -> real cost table
BENCH = [adhoc("bench_cpu_L3", "cpu", 3), adhoc("bench_cpu_L4", "cpu", 4),
         adhoc("bench_gpu_L3", "gpu", 3), adhoc("bench_gpu_L4", "gpu", 4),
         adhoc("bench_gpu_L6", "gpu", 6), adhoc("bench_gpu_L9", "gpu", 9)]
wait(AJ.submit(BENCH, extra_args="--benchmark 20", usd_override=0.08, tag_stage="bench"))
AJ.sync()
tps = plan.load_bench()
print("measured tok/s:", {f"{d}-L{L}": round(v) for (d, L), v in sorted(tps.items())})
missing = [b["name"] for b in BENCH if (("gpu" if "gpu" in b["name"] else "cpu"), b["L"]) not in tps]
print("missing benchmarks:", missing or "none")
!{PY} plan.py --budget --l6-tpp 120
!{PY} plan.py --budget --l6-tpp 60

In [ ]:
import time
def wait(jobs, poll=60):
    ml = AJ.client()
    while True:
        st = {j: ml.jobs.get(j).status for j in jobs}
        print(time.strftime("%H:%M"), st, flush=True)
        if all(s in ("Completed", "Failed", "Canceled") for s in st.values()):
            return st
        time.sleep(poll)

def adhoc(name, device, L=3, stage=0):
    return dict(name=name, stage=stage, L=L, tpp=120, sched="cos", method="scaled",
                lr_mult=1.0, prod_mult=1.0, seed=0, device=device)

SMOKE = [adhoc("smoke_cpu", "cpu"), adhoc("smoke_gpu", "gpu")]
j1 = AJ.submit(SMOKE, pool_asset="azureml:wdladder-pool:smoke", extra_args="--smoke --steps 300 --stop-at-step 150",
               usd_override=0.05, tag_stage="smoke")
wait(j1)
j2 = AJ.submit(SMOKE, pool_asset="azureml:wdladder-pool:smoke", extra_args="--smoke --steps 300",
               usd_override=0.05, tag_stage="smoke")
wait(j2)
for r in SMOKE:
    s = AJ.read_remote_json(r["name"], "summary.json") or {}
    print(r["name"], "status", s.get("status"), "segments", s.get("segments"), "device", s.get("device"),
          "bpb", (s.get("final") or {}).get("bpb"))

## 8. Throughput benchmarks -> real budget table (~20 min, ~$0.5)
This measures tokens/s for each rung on each device. `plan.py --budget` then replaces the placeholders with real
numbers.

**CHECK BEFORE CONTINUING (the most important cost check):**
- Every `bench_*` result is present.
- The `$ cum` column at stage 4 is within your budget.
- Stage 5 at TPP 60 fits the remaining budget; otherwise use `--l6-tpp 30` (`L6_TPP` below).
- Stage 6 (L9) is optional. Run it only if the remaining budget is more than its row.
- If CPU L4 is cheaper than GPU L4 per run, set `C.PLACEMENT[4]="cpu"` in config.py (more CPU offload).

In [ ]:
BENCH = [adhoc("bench_cpu_L3", "cpu", 3), adhoc("bench_cpu_L4", "cpu", 4),
         adhoc("bench_gpu_L3", "gpu", 3), adhoc("bench_gpu_L4", "gpu", 4),
         adhoc("bench_gpu_L6", "gpu", 6), adhoc("bench_gpu_L9", "gpu", 9)]
wait(AJ.submit(BENCH, extra_args="--benchmark 20", usd_override=0.08, tag_stage="bench"))
AJ.sync()
!python plan.py --budget --l6-tpp 60
!python plan.py --budget --l6-tpp 30

In [ ]:
L6_TPP = 60          # <- set after reading the table (60 preferred, 30 if budget is tight)

## 9. Stage 1 (L3, CPU cluster): Muon-SW vs constant decay at MAI's law, 2 seeds + TPP30 (~$4)
Six runs (3 nodes x 2 runs). Each run checkpoints every 15 min and on SIGTERM, and resumes automatically when
resubmitted.

**CHECK:** six runs are listed as `active:`.

In [ ]:
AJ.sync()
AJ.submit(plan.stage1(), tag_stage=1)
AJ.status(plan.stage1())

### 9b. Check-in cell: run once or twice a day while any stage is in flight (~1 min)
This syncs results, resubmits pre-empted or crashed runs (they resume from their last checkpoint), and prints
spend. Use the same cell for later stages by changing `STAGES`.
If you keep the instance on anyway, you can instead run the watchdog in a terminal:
`nohup python azure_jobs.py supervise --stages 1 --interval 20 > ~/sup.log 2>&1 &`

**Early-kill check (saves money):**
- `interim_gap` shows the paired const-minus-scaled BPB at matched steps.
- If the gap is negative at 50% and 70% of training for **both** seeds, Gate 1 will FAIL.
- In that case run `AJ.cancel(1)` and go to step 10 with what has finished. The checkpoints are kept.

In [ ]:
STAGES = [1]
AJ.supervise(STAGES, l6_tpp=L6_TPP, once=True)
rows = analysis.load_runs()
print("interim gap (const - scaled bpb) by seed/step:", analysis.interim_gap(rows))
lg = plan.ledger_load(); print("spend so far:", plan.actual_spend(instance_hours=lg.get("instance_hours", 0)))

## 10. GATE 1 (automatic, pre-registered) + GPT-6.1 anomaly review
| Decision | Meaning | Next |
|---|---|---|
| **PASS** | Muon-SW beats constant decay by more than max(0.003 bpb, 2 x seed noise) | stage 2 + stage 4 |
| **WEAK** | gain is positive but inside noise | stage 2 only; no GPU until gate 2 |
| **FAIL** | no gain at MAI hyperparameters | **STOP.** Write the non-replication note (O3) with `analysis --stage 1` + `llm_review draft`; about $10 spent in total |

**CHECK BEFORE CONTINUING:**
- Read `report/report.md`.
- Read the LLM review. It may flag anomalies such as a diverged seed or a gain driven by one seed. Resolve those before spending more.

In [ ]:
AJ.sync()
!python analysis.py --stage 1
!python llm_review.py review --stage 1

## 11. Stage 2 (L3 2-D grid in MAI coordinates, CPU) [+ stage 4 L4 on GPU if PASS] (~$26 + ~$24)
Stage 2 sweeps `(ln LR, ln LR*WD)` as multiples of MAI's law: 3 x 5 points per method. It also adds TPP60 and TPP240
reference runs for the compute-equivalent speed-up law.
Use the check-in cell with `STAGES=[2,4]` (or `[2]` if WEAK).

After stage 2 finishes, `analysis --stage 2` fits 2-D quadratics with bootstrap CIs. **Edge rule:** if an optimum
lies at the grid edge, it writes `edge_ext.json`. Re-running the next cell submits only the extension points.
Repeat at most twice.

**CHECK BEFORE CONTINUING:**
- `edge_extension: []`.
- Both fits are convex: `convex: true`, with R² above 0.9.
- `wd_ratio_cos` has a finite CI.
- The LLM review shows no unresolved issues.

In [ ]:
G1 = json.load(open(os.path.join(C.P["report"], "results.json")))["gate1"]["decision"]
print("gate 1:", G1)
assert G1 in ("PASS", "WEAK"), "gate 1 did not pass - stop here (see step 10)"
STAGES = [2, 4] if G1 == "PASS" else [2]
AJ.sync()
AJ.submit(AJ.stage_runs(STAGES, L6_TPP), tag_stage=2)

In [ ]:
# after all stage-2 runs are done (check-in cell with STAGES above):
AJ.sync()
!python analysis.py --stage 2
!python llm_review.py review --stage 2
ext = json.load(open(os.path.join(C.P["report"], "edge_ext.json")))
if ext:
    print("edge extension needed:", ext); AJ.submit(AJ.stage_runs([2], L6_TPP), tag_stage=2)

## 12. Stage 3 (L3 confirmation at fitted optima, seeds 1-2, + WSD 1-D) -> GATE 2 (~$12)
The **tuned gap** with its CI decides the paper's headline:
- **O1 genuine:** Muon-SW keeps X% of its gain after both methods are tuned.
- **O2 calibration artifact:** the gain disappears; MAI's law plus a fitted constant multiplier is enough.
- **O4 reversal:** tuned constant decay wins.

All three outcomes are publishable; see the earlier outcome table.

**CHECK BEFORE CONTINUING:**
- `tuned_gap` and `survival_fraction` are printed.
- `hypothesis_verdict` names which of H_none, H_int or H_end is consistent with the measured WD ratio.
- **Spend gate:** continue to stage 5 only if `remaining budget > stage-5 row` in `plan.py --budget`.
  Otherwise write up at L3/L4; the paper still stands.

In [ ]:
AJ.sync()
AJ.submit(AJ.stage_runs([3], L6_TPP), tag_stage=3)
# ... check-in cell with STAGES=[3] until done, then:

In [ ]:
AJ.sync()
!python analysis.py --stage 3
!python llm_review.py review --stage 3
!python plan.py --budget --l6-tpp {L6_TPP}

## 13. Stage 5 (L6 on the T4): does the fitted WD ratio transfer up the ladder? (~$45-86)
This tests whether the stage-2/3 ratio, predicted for L6 from L3 and L4, lands on the L6 optimum. That is the
ladder-transfer claim at the core of MAI's method. Runs are ordered so the most informative points go first; you can
cancel the rest with `AJ.cancel(5)` if the budget runs low.

**CHECK BEFORE CONTINUING:** `L6` in the output shows predicted vs measured optimum with CI, and the budget remains.

In [ ]:
AJ.sync()
AJ.submit(AJ.stage_runs([5], L6_TPP), tag_stage=5)
# check-in cell with STAGES=[5]; then:

In [ ]:
AJ.sync()
!python analysis.py --stage 5
!python llm_review.py review --stage 5

## 14. Optional stage 6 (L9 TPP30, 2 runs on the T4)
Run this only if `plan.py --budget` shows enough remaining budget. It gives one larger-rung point: constant decay
at MAI's law vs Muon-SW at its predicted optimum.

In [ ]:
# AJ.submit(AJ.stage_runs([6], L6_TPP), tag_stage=6)
# later: AJ.sync(); !python analysis.py --stage 6

## 15. Report, draft, spend ledger, cleanup
- `report/report.md`, `report/results.json` and `report/fig_*.png` are the paper's results.
- `report/results_draft.tex` is GPT-6.1's Results draft. Every number in it is checked against results.json, and
  any unverified number is listed at the bottom. **Hand-check them.**
- Log instance hours (Azure portal > Cost analysis, or your own count) so the ledger is complete.

**CHECK:**
- No active jobs remain.
- The GPU cluster has scaled to 0 nodes.
- **The compute instance is stopped.**

In [ ]:
FINAL_STAGE = 5          # highest stage you ran
!python analysis.py --stage {FINAL_STAGE}
!python llm_review.py draft
lg = plan.ledger_load(); lg["instance_hours"] = 0.0   # <- enter total instance hours used
plan.ledger_save(lg)
print(plan.actual_spend(instance_hours=lg["instance_hours"]))
print("active jobs:", AJ.active_runs())